In [1]:
import sys; print(sys.executable)

/home/ron/Documents/Github/VLM-lens/dl/bin/python


In [2]:
import json

with open('/home/ron/Documents/Github/VLM-lens/results/baseline/predictions.json', 'r') as file:
    data = json.load(file)

In [3]:
import pandas as pd
df = pd.json_normalize(data)

In [4]:
df.head()

,question_idx,scene_id,question,question_type,ground_truth,prediction,generated_text,correct,is_held_out_combo,image_path
0,0,0,What color is the circle?,query_color_unambiguous,red,red,The circle is red.,True,True,val/images/00000000.png
1,1,0,What shape is the red object?,query_shape_unambiguous,circle,circle,The red object is a circle.,True,True,val/images/00000000.png


In [4]:
df["question_type"].value_counts()

question_type
query_shape_unambiguous    205
query_color_unambiguous    144
query_color_negation        50
Name: count, dtype: int64

In [5]:
df[df["question_type"]=="query_color_negation"]["generated_text"].to_list()

['The object that is NOT yellow is blue.',
 'The object that is NOT blue is yellow.',
 'The object that is NOT blue is red.',
 'The object that is NOT red is blue.',
 'The object that is NOT blue is yellow.',
 'The object that is NOT yellow is blue.',
 'The object that is NOT purple is red.',
 'The object that is NOT red is purple.',
 'Yellow',
 'The object that is NOT yellow is black.',
 'The object that is NOT green is red.',
 'The object that is NOT red is green.',
 'The object that is NOT black is blue.',
 'The object that is NOT blue is black.',
 'The object that is NOT yellow is black.',
 'The object that is NOT black is yellow.',
 'The object that is NOT yellow is purple.',
 'The object that is NOT purple is yellow.',
 'The object that is NOT red is green.',
 'The object that is NOT green is red.',
 'The object that is NOT blue is purple.',
 'The object that is NOT purple is blue.',
 'The object that is NOT black is yellow.',
 'The object that is NOT yellow is black.',
 'The obj

In [6]:
df[df["generated_text"]=="The triangle is black and green."]

,question_idx,scene_id,question,question_type,ground_truth,prediction,generated_text,correct,is_held_out_combo,image_path
158,158,42,What color is the triangle?,query_color_unambiguous,black,black,The triangle is black and green.,True,True,val/images/00000042.png


In [10]:
df[["ground_truth", "prediction"]][df["ground_truth"]=="cyan"][df["prediction"]=="blue"].shape

/tmp/ipykernel_29957/2497064745.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[["ground_truth", "prediction"]][df["ground_truth"]=="cyan"][df["prediction"]=="blue"].shape


(0, 2)

In [5]:
from sae.llava_clt.utils.loader_functions import load_model

model, processor = load_model()

/home/ron/Documents/Github/VLM-lens/dl/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: llava-hf/llava-1.5-7b-hf


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.14it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [7]:
print(model)

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
       

In [10]:
for i, layer in enumerate(model.model.language_model.layers):
    print(f"Layer {i}: {layer}")
# layers = []
# for i, layer in enumerate(model.model.language_model.layers):
#     layers.append((f"model.language_model.layers.{i}", layer))

Layer 0: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)
Layer 1: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (v_proj): Linear(in_features=4096, out_features=4096, bias=Fa

In [16]:
model.model.language_model.layers[0]

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)

In [23]:
LAYER_IDX = 0  # try 0, 8, 16, 24, 31 later
layer = model.model.language_model.layers[LAYER_IDX]

In [26]:
import torch
from PIL import Image
from sae.llava_clt.run_model import format_prompt

img_path = "/home/ron/Documents/Github/VLM-lens/data/val/images/00000000.png"
question = "What color is the circle?"
answer = "red"
raw_image = Image.open(img_path)

prompt = format_prompt(question, processor)

inputs = processor(text=prompt, images=raw_image, return_tensors="pt").to(model.device, dtype=torch.float16)

In [28]:
obs_cache = []
obs_handle = layer.register_forward_hook(
    lambda m, inp, out: (print(f"[observe] Layer {LAYER_IDX} output shape:", tuple(out.shape)),
                         obs_cache.append(out.detach().to("cpu"))) and out
)

with torch.no_grad():
    logits_obs = model(**inputs, use_cache=False).logits

[observe] Layer 0 output shape: (1, 594, 4096)
[observe] Layer 0 output shape: (1, 594, 4096)


In [29]:
topk = torch.topk(logits_obs[0, -1], k=5)
ids = topk.indices.tolist()
tokens = [processor.tokenizer.convert_ids_to_tokens(i) for i in ids]
print("\n[baseline] top-5 next tokens:", tokens)


[baseline] top-5 next tokens: ['▁The', '▁There', '▁In', '▁Both', '▁It']


In [30]:
import torch
model.eval()

# how many future steps and how many candidates
STEPS = 5
TOPK  = 5

# start from your existing 'inputs' dict built with the chat template (includes <image>)
cur_ids = inputs["input_ids"].clone()
cur_attn = inputs["attention_mask"].clone()
pixel_values = inputs["pixel_values"]  # keep same image features

with torch.no_grad():
    for step in range(1, STEPS + 1):
        out = model(
            input_ids=cur_ids,
            attention_mask=cur_attn,
            pixel_values=pixel_values,
            use_cache=False  # set True for speed if you manage past_key_values yourself
        )
        next_logits = out.logits[:, -1, :]            # [B=1, V]
        topk = torch.topk(next_logits, k=TOPK, dim=-1)
        ids = topk.indices[0].tolist()
        vals = topk.values[0].tolist()
        toks = [processor.tokenizer.convert_ids_to_tokens(i) for i in ids]

        print(f"\nStep {step} — top-{TOPK} candidates for the NEXT token:")
        for i, (tok, logit) in enumerate(zip(toks, vals), 1):
            print(f"  {i}. {tok!r:>12}  logit={logit:.3f}")

        # Greedy append the best next token (you can sample instead)
        next_token = topk.indices[:, :1]              # [1,1]
        cur_ids = torch.cat([cur_ids, next_token], dim=1)

        # extend attention mask
        one = torch.ones((cur_attn.size(0), 1), device=cur_attn.device, dtype=cur_attn.dtype)
        cur_attn = torch.cat([cur_attn, one], dim=1)

# Optional: decode the whole continuation you just appended (last STEPS tokens)
generated = processor.tokenizer.decode(cur_ids[0, -STEPS:].tolist(), skip_special_tokens=False)
print("\nGreedy continuation (last steps):", repr(generated))


[observe] Layer 0 output shape: (1, 594, 4096)
[observe] Layer 0 output shape: (1, 594, 4096)

Step 1 — top-5 candidates for the NEXT token:
  1.       '▁The'  logit=28.828
  2.     '▁There'  logit=21.781
  3.        '▁In'  logit=21.297
  4.      '▁Both'  logit=20.594
  5.        '▁It'  logit=19.594
[observe] Layer 0 output shape: (1, 595, 4096)
[observe] Layer 0 output shape: (1, 595, 4096)

Step 2 — top-5 candidates for the NEXT token:
  1.    '▁circle'  logit=23.984
  2.     '▁color'  logit=20.688
  3.  '▁circular'  logit=15.836
  4.       '▁red'  logit=15.609
  5.    '▁colors'  logit=14.938
[observe] Layer 0 output shape: (1, 596, 4096)
[observe] Layer 0 output shape: (1, 596, 4096)

Step 3 — top-5 candidates for the NEXT token:
  1.        '▁is'  logit=24.312
  2.        '▁in'  logit=21.484
  3.       '▁has'  logit=20.797
  4.   '▁appears'  logit=18.391
  5.     '▁color'  logit=18.188
[observe] Layer 0 output shape: (1, 597, 4096)
[observe] Layer 0 output shape: (1, 597, 4096)

St